# Automatic Lens Correction v2 — Improved Pipeline

**Key improvements over v1:**
1. Finer parameter extraction (21×13 grid + optimization)
2. Stricter outlier filtering (SSIM >= 0.75, clamp k1/k2)
3. Higher resolution training (384×384)
4. Per-image Hough line refinement at inference time
5. Mixed precision training for speed on GPU

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torch.amp import autocast, GradScaler

from scipy.optimize import minimize
from skimage.metrics import structural_similarity as ssim

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

# Detect environment
ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    print(f'Kaggle input contents: {os.listdir("/kaggle/input")}')
    
    # Competition data is at /kaggle/input/competitions/automatic-lens-correction/
    COMP_DIR = '/kaggle/input/competitions/automatic-lens-correction'
    if not os.path.exists(COMP_DIR):
        # Fallback: search for it
        COMP_DIR = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'automatic-lens-correction' in dirs:
                COMP_DIR = os.path.join(root, 'automatic-lens-correction')
                break
            # Stop after 2 levels
            if root.count(os.sep) - '/kaggle/input'.count(os.sep) >= 2:
                break
    
    print(f'Competition dir: {COMP_DIR}')
    if COMP_DIR:
        print(f'Contents: {os.listdir(COMP_DIR)}')
    
    # Find training and test dirs
    TRAIN_DIR = None
    TEST_DIR = None
    
    if COMP_DIR:
        # List all subdirs
        for item in os.listdir(COMP_DIR):
            path = os.path.join(COMP_DIR, item)
            if os.path.isdir(path):
                sample = os.listdir(path)[:10]
                print(f'  Subdir: {item}/ -> {sample[:5]}')
                
                if any('_original.jpg' in f for f in sample):
                    TRAIN_DIR = path
                elif 'test' in item.lower() or 'original' in item.lower():
                    TEST_DIR = path
        
        # If test not found, check for non-paired jpgs in remaining dirs
        if TEST_DIR is None:
            for item in os.listdir(COMP_DIR):
                path = os.path.join(COMP_DIR, item)
                if os.path.isdir(path) and path != TRAIN_DIR:
                    sample = os.listdir(path)[:10]
                    if any(f.endswith('.jpg') for f in sample):
                        TEST_DIR = path
    
    WORKING_DIR = '/kaggle/working'
    OUTPUT_DIR = '/kaggle/working/corrected'
else:
    # Local paths
    PROJECT_DIR = '/Users/saiteja/Documents/Dev/AutoHDR'
    TRAIN_DIR = os.path.join(PROJECT_DIR, 'lens-correction-train-cleaned')
    TEST_DIR = os.path.join(PROJECT_DIR, 'test-originals')
    WORKING_DIR = os.path.join(PROJECT_DIR, 'outputs')
    OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs', 'corrected_v2')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORKING_DIR, exist_ok=True)

print(f'\nTrain dir: {TRAIN_DIR}')
print(f'Test dir: {TEST_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# Parse training pairs
# First, make sure TRAIN_DIR exists and has files
if TRAIN_DIR is None:
    print('ERROR: TRAIN_DIR not found!')
    print('Listing everything in /kaggle/input recursively (2 levels):')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 3:
            indent = ' ' * 2 * depth
            print(f'{indent}{os.path.basename(root)}/ ({len(files)} files, {len(dirs)} dirs)')
            if files:
                print(f'{indent}  Sample files: {files[:5]}')
    raise FileNotFoundError('Could not find training directory')

print(f'Train dir: {TRAIN_DIR}')
train_files = sorted(os.listdir(TRAIN_DIR))
print(f'Total files: {len(train_files)}')
print(f'Sample: {train_files[:6]}')

pairs = defaultdict(dict)
for f in train_files:
    if not f.endswith('.jpg'):
        continue
    parts = f.rsplit('_', 1)
    if len(parts) == 2:
        image_id = parts[0]
        img_type = parts[1].replace('.jpg', '')
        pairs[image_id][img_type] = os.path.join(TRAIN_DIR, f)

complete_pairs = {k: v for k, v in pairs.items() if 'original' in v and 'generated' in v}
print(f'Complete training pairs: {len(complete_pairs)}')

if len(complete_pairs) == 0:
    print('WARNING: No pairs found! Files might not follow expected naming pattern.')
    print(f'Expected pattern: {{uuid}}_{{gN}}_original.jpg / _generated.jpg')
    print(f'Actual files: {train_files[:10]}')

## 1. Improved Parameter Extraction

**Changes from v1:**
- Finer grid: 21 k1 values × 13 k2 values (vs 11×7)
- Extended k2 range to [-0.5, 0.5] (vs [-0.3, 0.3])
- More optimization iterations (300 vs 200)

In [ ]:
def extract_distortion_params_v2(original_path, generated_path, resize_dim=256):
    """
    Improved parameter extraction with finer grid search.
    """
    orig = cv2.imread(original_path)
    gen = cv2.imread(generated_path)
    
    if orig is None or gen is None:
        return None, None, 0.0
    
    orig_small = cv2.resize(orig, (resize_dim, resize_dim))
    gen_small = cv2.resize(gen, (resize_dim, resize_dim))
    
    h, w = orig_small.shape[:2]
    fx = fy = w
    cx, cy = w / 2.0, h / 2.0
    camera_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    
    gen_gray = cv2.cvtColor(gen_small, cv2.COLOR_BGR2GRAY)
    
    def objective(params):
        k1, k2 = params
        dist_coeffs = np.array([k1, k2, 0, 0, 0], dtype=np.float64)
        undistorted = cv2.undistort(orig_small, camera_matrix, dist_coeffs)
        undist_gray = cv2.cvtColor(undistorted, cv2.COLOR_BGR2GRAY)
        score = ssim(gen_gray, undist_gray)
        return -score
    
    # FINER grid search: 21 x 13 = 273 combinations (vs 77 in v1)
    best_score = -1
    best_k1, best_k2 = 0.0, 0.0
    
    for k1 in np.arange(-0.5, 0.51, 0.05):    # 21 steps (vs 11)
        for k2 in np.arange(-0.5, 0.51, 0.08):  # 13 steps (vs 7), wider range
            score = -objective([k1, k2])
            if score > best_score:
                best_score = score
                best_k1, best_k2 = k1, k2
    
    # Fine optimization with more iterations
    result = minimize(objective, [best_k1, best_k2],
                      method='Nelder-Mead',
                      options={'xatol': 1e-5, 'fatol': 1e-7, 'maxiter': 300})
    
    k1_opt, k2_opt = result.x
    final_ssim = -result.fun
    
    return k1_opt, k2_opt, final_ssim

print('Extraction function defined.')

In [ ]:
# Extract all parameters (or load cached)
PARAMS_CSV = os.path.join(WORKING_DIR, 'distortion_params_v2.csv')

# Check multiple locations for existing CSV
csv_candidates = [
    PARAMS_CSV,
    os.path.join(WORKING_DIR, 'distortion_params.csv'),
]
if ON_KAGGLE:
    # Check uploaded dataset (both possible paths)
    csv_candidates.insert(0, '/kaggle/input/datasets/saitejacore/autohdr-distortion-params/distortion_params.csv')
    csv_candidates.insert(1, '/kaggle/input/autohdr-distortion-params/distortion_params.csv')

print('Searching for CSV in:')
for p in csv_candidates:
    exists = os.path.exists(p)
    print(f'  {p} -> {"FOUND" if exists else "not found"}')

loaded = False
for csv_path in csv_candidates:
    if os.path.exists(csv_path):
        print(f'\nLoading parameters from {csv_path}')
        params_df = pd.read_csv(csv_path)
        loaded = True
        break

if not loaded:
    print(f'\nNo cached CSV found. Extracting parameters for {len(complete_pairs)} pairs...')
    print('This will take ~2-3 hours...')
    results = []
    pair_ids = list(complete_pairs.keys())

    for i, img_id in enumerate(tqdm(pair_ids)):
        k1, k2, score = extract_distortion_params_v2(
            complete_pairs[img_id]['original'],
            complete_pairs[img_id]['generated']
        )
        results.append({
            'image_id': img_id,
            'original_path': complete_pairs[img_id]['original'],
            'generated_path': complete_pairs[img_id]['generated'],
            'k1': k1, 'k2': k2, 'ssim_score': score
        })
        if (i + 1) % 500 == 0:
            pd.DataFrame(results).to_csv(PARAMS_CSV, index=False)
            print(f'  Saved {i+1}/{len(pair_ids)}')

    params_df = pd.DataFrame(results)
    params_df.to_csv(PARAMS_CSV, index=False)
    print(f'Saved to {PARAMS_CSV}')

print(f'\nTotal samples: {len(params_df)}')
print(params_df[['k1', 'k2', 'ssim_score']].describe())

In [ ]:
# STRICT filtering — remove noisy labels
valid_params = params_df.dropna(subset=['k1', 'k2']).copy()

before = len(valid_params)
valid_params = valid_params[valid_params['ssim_score'] >= 0.75]
valid_params = valid_params[valid_params['k1'].abs() <= 0.4]
valid_params = valid_params[valid_params['k2'].abs() <= 0.8]
after = len(valid_params)

# Fix file paths if CSV was created on a different machine
if ON_KAGGLE and 'original_path' in valid_params.columns:
    if '/Users/' in str(valid_params['original_path'].iloc[0]):
        print('Fixing file paths for Kaggle environment...')
        valid_params['original_path'] = valid_params['image_id'].apply(
            lambda x: os.path.join(TRAIN_DIR, x + '_original.jpg'))
        valid_params['generated_path'] = valid_params['image_id'].apply(
            lambda x: os.path.join(TRAIN_DIR, x + '_generated.jpg'))

print(f'Kept {after} / {before} samples ({100*after/before:.1f}%)')
print(f'Dropped {before - after} noisy samples')
print(f'\nFiltered stats:')
print(valid_params[['k1', 'k2', 'ssim_score']].describe())

## 2. Model Definition — ResNet50 + Higher Resolution

**Changes from v1:**
- Training at 384×384 (vs 256×256) — sees finer distortion details
- Stronger augmentation
- Mixed precision training (faster on GPU)

In [ ]:
IMG_SIZE = 384  # v1 used 256

class DistortionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['original_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        label = torch.tensor([row['k1'], row['k2']], dtype=torch.float32)
        return img, label


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f'Training resolution: {IMG_SIZE}x{IMG_SIZE}')

In [ ]:
# Train/val split by UUID
valid_params['uuid'] = valid_params['image_id'].apply(lambda x: x.rsplit('_', 1)[0])
unique_uuids = valid_params['uuid'].unique()
np.random.seed(42)
np.random.shuffle(unique_uuids)

split_idx = int(len(unique_uuids) * 0.9)
train_uuids = set(unique_uuids[:split_idx])
val_uuids = set(unique_uuids[split_idx:])

train_df = valid_params[valid_params['uuid'].isin(train_uuids)].copy()
val_df = valid_params[valid_params['uuid'].isin(val_uuids)].copy()

print(f'Train: {len(train_df)} | Val: {len(val_df)}')

BATCH_SIZE = 16 if IMG_SIZE >= 384 else 32

train_dataset = DistortionDataset(train_df, transform=train_transform)
val_dataset = DistortionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2 if ON_KAGGLE else 0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2 if ON_KAGGLE else 0, pin_memory=True)

print(f'Batch size: {BATCH_SIZE}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
class DistortionRegressor(nn.Module):
    def __init__(self, num_params=2):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        
        # Load pretrained weights from uploaded dataset or cache
        weight_paths = [
            '/kaggle/input/resnet50-pretrained-weights/resnet50.pth',
            '/kaggle/input/datasets/saitejacore/resnet50-pretrained-weights/resnet50.pth',
        ]
        loaded = False
        for wp in weight_paths:
            if os.path.exists(wp):
                self.backbone.load_state_dict(torch.load(wp, weights_only=True))
                print(f'Loaded pretrained weights from {wp}')
                loaded = True
                break
        
        if not loaded:
            try:
                self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
                print('Loaded pretrained weights from internet')
            except Exception as e:
                print(f'No pretrained weights available, using random init')
        
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_params)
        )
    
    def forward(self, x):
        return self.backbone(x)


model = DistortionRegressor(num_params=2).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Training functions with mixed precision
use_amp = (device.type == 'cuda')  # AMP only works well on CUDA
scaler = GradScaler('cuda') if use_amp else None

def freeze_backbone(model):
    for name, param in model.backbone.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False

def unfreeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = True

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        
        if use_amp:
            with autocast('cuda'):
                preds = model(images)
                loss = criterion(preds, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(images)
            loss = criterion(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            preds = model(images)
            loss = criterion(preds, labels)
            total_loss += loss.item()
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    mae = torch.abs(all_preds - all_labels).mean(dim=0)
    return total_loss / len(loader), mae, all_preds, all_labels

print(f'Mixed precision: {use_amp}')
print(f'Gradient clipping: max_norm=1.0')

## 3. Training

In [ ]:
# Phase 1: Head only
print('=' * 60)
print('PHASE 1: Head only (backbone frozen)')
print('=' * 60)

freeze_backbone(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable:,}')

criterion = nn.HuberLoss(delta=0.1)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

PHASE1_EPOCHS = 15
best_val_loss = float('inf')
phase1_path = os.path.join(WORKING_DIR, 'model_phase1_v2.pth')

for epoch in range(PHASE1_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_mae, _, _ = validate(model, val_loader, criterion)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']
    
    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), phase1_path)
        improved = ' ** BEST **'
    
    print(f'Epoch {epoch+1:2d}/{PHASE1_EPOCHS} | '
          f'Train: {train_loss:.6f} | Val: {val_loss:.6f} | '
          f'k1 MAE: {val_mae[0]:.4f} | k2 MAE: {val_mae[1]:.4f} | '
          f'LR: {lr:.1e}{improved}')

print(f'\nPhase 1 done. Best val loss: {best_val_loss:.6f}')

In [ ]:
# Phase 2: Full fine-tune
print('=' * 60)
print('PHASE 2: Full fine-tune (all layers)')
print('=' * 60)

model.load_state_dict(torch.load(phase1_path, weights_only=True))
unfreeze_backbone(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable:,}')

optimizer = optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

PHASE2_EPOCHS = 40
best_val_loss = float('inf')
best_path = os.path.join(WORKING_DIR, 'model_best_v2.pth')
patience_counter = 0
PATIENCE = 10

for epoch in range(PHASE2_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_mae, _, _ = validate(model, val_loader, criterion)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    
    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)
        improved = ' ** BEST **'
        patience_counter = 0
    else:
        patience_counter += 1
    
    print(f'Epoch {epoch+1:2d}/{PHASE2_EPOCHS} | '
          f'Train: {train_loss:.6f} | Val: {val_loss:.6f} | '
          f'k1 MAE: {val_mae[0]:.4f} | k2 MAE: {val_mae[1]:.4f} | '
          f'LR: {lr:.1e}{improved}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
        break

print(f'\nPhase 2 done. Best val loss: {best_val_loss:.6f}')

In [ ]:
# Final evaluation
model.load_state_dict(torch.load(best_path, weights_only=True))
val_loss, val_mae, preds, labels = validate(model, val_loader, criterion)

print(f'Final Performance:')
print(f'  k1 MAE: {val_mae[0]:.4f}')
print(f'  k2 MAE: {val_mae[1]:.4f}')
print(f'  Val Loss: {val_loss:.6f}')

## 4. Inference with Hough Line Refinement

**The key improvement:** After CNN predicts k1, k2, we refine per-image by testing
nearby values and picking the one that produces the straightest lines.

This directly optimizes for the Line Straightness metric (22% of score).

In [ ]:
def compute_line_straightness(image):
    """
    Score how straight the lines are in an image using Hough transform.
    Higher score = straighter lines = better correction.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (512, 384))
    
    # Edge detection
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 1.5), 50, 150)
    
    # Detect lines
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50,
                            minLineLength=40, maxLineGap=10)
    
    if lines is None or len(lines) < 5:
        return 0.0
    
    # Compute angles of all detected lines
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
        # Normalize to [0, 180)
        angle = angle % 180
        angles.append(angle)
    
    angles = np.array(angles)
    
    # Score: lines in real estate should be mostly horizontal (0/180) or vertical (90)
    # Measure how many lines are close to 0, 90, or 180 degrees
    h_mask = (angles < 15) | (angles > 165)  # horizontal
    v_mask = (angles > 75) & (angles < 105)   # vertical
    aligned = (h_mask | v_mask).sum()
    
    score = aligned / len(angles)
    return score


def correct_image_with_refinement(model, image_path, device, transform, refine=True):
    """
    Correct image with optional Hough line refinement.
    
    1. CNN predicts initial k1, k2
    2. If refine=True, search nearby values for best line straightness
    3. Apply best k1, k2 with cv2.undistort()
    """
    img_full = cv2.imread(image_path)
    if img_full is None:
        return None
    
    h, w = img_full.shape[:2]
    
    # CNN prediction
    img_rgb = cv2.cvtColor(img_full, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_tensor = transform(img_pil).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        pred = model(img_tensor).cpu().numpy()[0]
    
    k1_cnn, k2_cnn = pred[0], pred[1]
    
    # Camera matrix
    fx = fy = float(w)
    cx, cy = w / 2.0, h / 2.0
    camera_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    
    best_k1, best_k2 = k1_cnn, k2_cnn
    
    if refine:
        # Search grid around CNN prediction
        best_score = -1
        
        # Use smaller image for speed
        img_small = cv2.resize(img_full, (512, 384))
        h_s, w_s = img_small.shape[:2]
        cam_small = np.array([[w_s, 0, w_s/2], [0, w_s, h_s/2], [0, 0, 1]], dtype=np.float64)
        
        for dk1 in np.arange(-0.05, 0.06, 0.025):
            for dk2 in np.arange(-0.1, 0.11, 0.025):
                k1_try = k1_cnn + dk1
                k2_try = k2_cnn + dk2
                
                dist = np.array([k1_try, k2_try, 0, 0, 0], dtype=np.float64)
                undist = cv2.undistort(img_small, cam_small, dist)
                
                score = compute_line_straightness(undist)
                if score > best_score:
                    best_score = score
                    best_k1, best_k2 = k1_try, k2_try
    
    # Apply final correction at full resolution
    dist_coeffs = np.array([best_k1, best_k2, 0, 0, 0], dtype=np.float64)
    
    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dist_coeffs, (w, h), alpha=0, newImgSize=(w, h)
    )
    
    corrected = cv2.undistort(img_full, camera_matrix, dist_coeffs, None, new_camera_matrix)
    
    if roi != (0, 0, 0, 0):
        x, y, rw, rh = roi
        corrected = corrected[y:y+rh, x:x+rw]
        corrected = cv2.resize(corrected, (w, h), interpolation=cv2.INTER_LANCZOS4)
    
    return corrected, best_k1, best_k2, k1_cnn, k2_cnn


print('Correction with Hough refinement defined.')
print('Refinement grid: 5 k1 values x 9 k2 values = 45 trials per image')

In [ ]:
# Run inference on test images
model.load_state_dict(torch.load(best_path, weights_only=True))
model.eval()

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))])
print(f'Correcting {len(test_files)} test images with Hough refinement...')

results = []
for f in tqdm(test_files):
    img_path = os.path.join(TEST_DIR, f)
    result = correct_image_with_refinement(model, img_path, device, val_transform, refine=True)
    
    if result is not None:
        corrected, k1_final, k2_final, k1_cnn, k2_cnn = result
        out_path = os.path.join(OUTPUT_DIR, f)
        cv2.imwrite(out_path, corrected, [cv2.IMWRITE_JPEG_QUALITY, 95])
        results.append({
            'filename': f,
            'k1_cnn': k1_cnn, 'k2_cnn': k2_cnn,
            'k1_refined': k1_final, 'k2_refined': k2_final
        })

print(f'\nSaved {len(results)} corrected images to {OUTPUT_DIR}')

# Show how much refinement changed the predictions
res_df = pd.DataFrame(results)
res_df['k1_shift'] = (res_df['k1_refined'] - res_df['k1_cnn']).abs()
res_df['k2_shift'] = (res_df['k2_refined'] - res_df['k2_cnn']).abs()
print(f'\nRefinement shifts:')
print(f'  k1 avg shift: {res_df["k1_shift"].mean():.4f}')
print(f'  k2 avg shift: {res_df["k2_shift"].mean():.4f}')
print(f'  Images where refinement changed k2 by >0.05: {(res_df["k2_shift"] > 0.05).sum()}')

res_df.to_csv(os.path.join(WORKING_DIR, 'predictions_v2.csv'), index=False)

In [ ]:
# Create submission zip
import zipfile

ZIP_PATH = os.path.join(WORKING_DIR, 'corrected_images_v2.zip')
corrected_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.jpg')))

if corrected_files:
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in tqdm(corrected_files, desc='Zipping'):
            zf.write(f, os.path.basename(f))
    
    zip_size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
    print(f'\nCreated {ZIP_PATH}')
    print(f'Size: {zip_size_mb:.1f} MB')
    print(f'Images: {len(corrected_files)}')
    
    # If too large, resave at lower quality
    if zip_size_mb > 490:
        print(f'\nZip too large! Resaving at quality 85...')
        for f in tqdm(corrected_files):
            img = cv2.imread(f)
            cv2.imwrite(f, img, [cv2.IMWRITE_JPEG_QUALITY, 85])
        
        with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in corrected_files:
                zf.write(f, os.path.basename(f))
        
        zip_size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
        print(f'New size: {zip_size_mb:.1f} MB')

print('\nNext: Upload zip to bounty.autohdr.com, download submission.csv, submit to Kaggle')

In [ ]:
# Save final model
final_path = os.path.join(WORKING_DIR, 'final_model_v2.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'ResNet50_DistortionRegressor',
    'num_params': 2,
    'input_size': IMG_SIZE,
    'normalization': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
    'uses_hough_refinement': True,
}, final_path)

print(f'Model saved to {final_path}')
print('\n--- DONE ---')